# VexarDrive Fleet Analysis

## Vehicle Health

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROCESSED_DIR = Path("../data/processed")

vehicles = pd.read_parquet(PROCESSED_DIR / "vehicles.parquet")
trips = pd.read_parquet(PROCESSED_DIR / "trips.parquet")
telemetry = pd.read_parquet(PROCESSED_DIR / "telemetry.parquet")

vehicles.shape, trips.shape, telemetry.shape

((30, 8), (450, 14), (12987, 13))

In [3]:
vehicles.head()

,Vehicle_ID,Vehicle_Type,Make,Model,Manufacture_Year,Registration_Date,Odometer_KM_Start_of_Week,Last_Service_Date
0,V01,Two-Wheeler (Scooter/Motorcycle),Yamaha,Ray ZR,2021,2021-04-23,22820,2026-06-08
1,V02,Two-Wheeler (Scooter/Motorcycle),TVS,Raider,2021,2021-03-26,46601,2026-05-06
2,V03,Two-Wheeler (Scooter/Motorcycle),Yamaha,Ray ZR,2022,2022-11-07,14883,2026-07-08
3,V04,Two-Wheeler (Scooter/Motorcycle),Honda,Activa,2020,2020-07-02,5533,2026-07-19
4,V05,Two-Wheeler (Scooter/Motorcycle),TVS,Raider,2025,2025-02-25,2727,2026-07-12


In [4]:
trips.head()

,Trip_ID,Driver_ID,Vehicle_ID,Trip_Date,Start_Time,End_Time,Duration_Min,Distance_KM,Avg_Speed_kmph,Max_Speed_kmph,Start_Latitude,Start_Longitude,End_Latitude,End_Longitude
0,T00001,D01,V01,2026-08-04,11:08:00,11:24:00,16,9.34,28.7,39.4,12.991693,77.559009,13.011179,77.607400
1,T00002,D01,V01,2026-08-02,20:38:00,21:00:00,22,5.00,28.5,54.4,13.000790,77.560607,13.038001,77.563879
2,T00003,D01,V01,2026-07-31,21:11:00,21:38:00,27,7.25,16.9,32.7,12.983049,77.554049,12.949627,77.521240
3,T00004,D01,V01,2026-07-31,17:36:00,18:18:00,42,9.27,23.9,40.4,12.974957,77.549494,12.987157,77.479689
4,T00005,D01,V01,2026-08-01,17:06:00,17:19:00,13,7.74,28.7,42.5,12.997466,77.547647,12.997395,77.504517


In [5]:
telemetry.head()

,Trip_ID,Driver_ID,Vehicle_ID,Timestamp,Latitude,Longitude,Speed_kmph,Accel_X_g,Accel_Y_g,Accel_Z_g,Gyro_X_dps,Gyro_Y_dps,Gyro_Z_dps
0,T00001,D01,V01,2026-08-04 11:08:00,12.991693,77.559009,22.1,0.070,-0.035,0.998,1.49,1.09,-1.33
1,T00001,D01,V01,2026-08-04 11:09:00,12.993590,77.564465,37.9,0.035,0.013,0.968,-0.64,-0.94,-1.28
2,T00001,D01,V01,2026-08-04 11:10:00,12.993377,77.568199,20.0,-0.096,0.029,1.049,-0.70,-0.92,1.72
3,T00001,D01,V01,2026-08-04 11:11:00,12.993633,77.573369,27.9,-0.505,0.020,1.027,-0.62,0.91,-1.32
4,T00001,D01,V01,2026-08-04 11:12:00,12.992679,77.576146,23.6,0.017,0.046,0.982,-0.16,-3.37,-2.89


## Vehicle Coverage

Check how much telemetry and how many trips are available for each vehicle.

In [6]:
vehicle_trip_counts = (
    trips.groupby("Vehicle_ID")
    .agg(
        Trip_Count=("Trip_ID", "nunique")
    )
    .reset_index()
)

vehicle_trip_counts.describe()

,Trip_Count
count,30.000000
mean,15.000000
std,1.508596
min,9.000000
25%,15.000000
50%,15.000000
75%,16.000000
max,18.000000


In [7]:
vehicle_telemetry_counts = (
    telemetry.groupby("Vehicle_ID")
    .size()
    .reset_index(name="Telemetry_Rows")
)

vehicle_telemetry_counts.describe()

,Telemetry_Rows
count,30.000000
mean,432.900000
std,55.222028
min,293.000000
25%,401.750000
50%,438.500000
75%,465.000000
max,555.000000


## Acceleration Behaviour

In [8]:
accel_columns = [
    "Accel_X_g",
    "Accel_Y_g",
    "Accel_Z_g",
]

telemetry[accel_columns].describe()

,Accel_X_g,Accel_Y_g,Accel_Z_g
count,12987.000000,12987.000000,12987.000000
mean,0.001730,0.000527,1.006644
std,0.156097,0.086824,0.076165
min,-0.825000,-0.578000,0.763000
25%,-0.042000,-0.039000,0.974000
50%,0.000000,0.000000,1.001000
75%,0.042000,0.040000,1.028000
max,0.898000,0.579000,1.869000


In [9]:
telemetry[accel_columns].isna().sum()

Accel_X_g    0
Accel_Y_g    0
Accel_Z_g    0
dtype: int64

## Gyroscope Behaviour

In [10]:
gyro_columns = [
    "Gyro_X_dps",
    "Gyro_Y_dps",
    "Gyro_Z_dps",
]

telemetry[gyro_columns].describe()

,Gyro_X_dps,Gyro_Y_dps,Gyro_Z_dps
count,12987.000000,12987.000000,12987.000000
mean,-0.010265,-0.011844,0.057158
std,2.209659,2.199715,8.295250
min,-27.240000,-20.030000,-58.150000
25%,-1.360000,-1.365000,-1.400000
50%,-0.030000,-0.020000,-0.010000
75%,1.350000,1.360000,1.415000
max,19.270000,26.400000,56.500000


In [11]:
telemetry[gyro_columns].isna().sum()

Gyro_X_dps    0
Gyro_Y_dps    0
Gyro_Z_dps    0
dtype: int64

## Sensor Features

In [13]:
telemetry["Accel_Horizontal_g"] = np.sqrt(
    telemetry["Accel_X_g"] ** 2
    + telemetry["Accel_Y_g"] ** 2
)

telemetry["Gyro_Magnitude_dps"] = np.sqrt(
    telemetry["Gyro_X_dps"] ** 2
    + telemetry["Gyro_Y_dps"] ** 2
    + telemetry["Gyro_Z_dps"] ** 2
)

telemetry[
    [
        "Accel_Horizontal_g",
        "Gyro_Magnitude_dps",
    ]
].describe()

,Accel_Horizontal_g,Gyro_Magnitude_dps
count,12987.000000,12987.000000
mean,0.112950,4.716034
std,0.138381,7.502889
min,0.001000,0.107238
25%,0.045398,2.255227
50%,0.072007,3.176240
75%,0.108173,4.241267
max,0.901556,58.206043


## Vehicle-Level Sensor Metrics

In [14]:
vehicle_sensor_metrics = (
    telemetry
    .groupby("Vehicle_ID")
    .agg(
        P95_Accel_Horizontal=(
            "Accel_Horizontal_g",
            lambda x: x.quantile(0.95)
        ),
        P99_Accel_Horizontal=(
            "Accel_Horizontal_g",
            lambda x: x.quantile(0.99)
        ),
        Mean_Accel_Horizontal=(
            "Accel_Horizontal_g",
            "mean"
        ),
        Std_Accel_Horizontal=(
            "Accel_Horizontal_g",
            "std"
        ),
        P95_Gyro=(
            "Gyro_Magnitude_dps",
            lambda x: x.quantile(0.95)
        ),
        P99_Gyro=(
            "Gyro_Magnitude_dps",
            lambda x: x.quantile(0.99)
        ),
        Mean_Gyro=(
            "Gyro_Magnitude_dps",
            "mean"
        ),
        Std_Gyro=(
            "Gyro_Magnitude_dps",
            "std"
        ),
    )
    .reset_index()
)

vehicle_sensor_metrics.head()

,Vehicle_ID,P95_Accel_Horizontal,P99_Accel_Horizontal,Mean_Accel_Horizontal,Std_Accel_Horizontal,P95_Gyro,P99_Gyro,Mean_Gyro,Std_Gyro
0,V01,0.494907,0.757024,0.122926,0.148481,29.445111,51.357618,5.442126,9.350456
1,V02,0.380486,0.597780,0.128973,0.111654,16.350469,44.948347,4.920702,6.963669
2,V03,0.609615,0.736601,0.150214,0.187903,28.911927,50.827764,5.390473,9.010858
3,V04,0.380365,0.600559,0.097189,0.114784,25.854231,46.188237,5.144587,8.308738
4,V05,0.142751,0.382663,0.077300,0.069323,6.457722,41.288460,4.008126,5.486347


In [15]:
vehicle_sensor_metrics.describe().T

,count,mean,std,min,25%,50%,75%,max
P95_Accel_Horizontal,30.0,0.397685,0.174491,0.142751,0.198369,0.427519,0.550182,0.641340
P99_Accel_Horizontal,30.0,0.641252,0.113144,0.380702,0.595540,0.681910,0.732952,0.763440
Mean_Accel_Horizontal,30.0,0.112045,0.024705,0.076940,0.093502,0.105341,0.127461,0.160259
Std_Accel_Horizontal,30.0,0.130350,0.036484,0.069323,0.098559,0.127005,0.163933,0.197637
P95_Gyro,30.0,14.463837,10.065445,5.769172,6.293561,7.520658,25.764854,32.722102
P99_Gyro,30.0,44.184303,6.432924,22.599570,42.227375,45.971265,48.573611,51.761746
Mean_Gyro,30.0,4.689411,0.594952,3.629004,4.098330,4.772243,5.185247,5.574218
Std_Gyro,30.0,7.283612,1.507779,4.451791,5.754882,7.709630,8.544164,9.360922


## Extreme Sensor Events

In [16]:
accel_threshold = telemetry["Accel_Horizontal_g"].quantile(0.95)
gyro_threshold = telemetry["Gyro_Magnitude_dps"].quantile(0.95)

accel_threshold, gyro_threshold

(np.float64(0.4649171969286574), np.float64(7.575581189337716))

In [17]:
vehicle_extreme_rates = (
    telemetry
    .groupby("Vehicle_ID")
    .agg(
        High_Accel_Rate=(
            "Accel_Horizontal_g",
            lambda x: (x > accel_threshold).mean()
        ),
        High_Gyro_Rate=(
            "Gyro_Magnitude_dps",
            lambda x: (x > gyro_threshold).mean()
        ),
    )
    .reset_index()
)

vehicle_extreme_rates.head()

,Vehicle_ID,High_Accel_Rate,High_Gyro_Rate
0,V01,0.060890,0.065574
1,V02,0.029345,0.083521
2,V03,0.114173,0.064961
3,V04,0.029474,0.058947
4,V05,0.008299,0.031120


In [18]:
vehicle_extreme_rates.describe().T

,count,mean,std,min,25%,50%,75%,max
High_Accel_Rate,30.0,0.048902,0.032859,0.005602,0.024999,0.040213,0.072459,0.121076
High_Gyro_Rate,30.0,0.049395,0.018588,0.014006,0.035058,0.050636,0.064004,0.083521


## Vehicle Health Features

Combine the vehicle-level sensor metrics into one table.

In [19]:
vehicle_features = (
    vehicle_sensor_metrics
    .merge(
        vehicle_extreme_rates,
        on="Vehicle_ID",
        how="left"
    )
    .merge(
        vehicle_trip_counts,
        on="Vehicle_ID",
        how="left"
    )
)

vehicle_features.head()

,Vehicle_ID,P95_Accel_Horizontal,P99_Accel_Horizontal,Mean_Accel_Horizontal,Std_Accel_Horizontal,P95_Gyro,P99_Gyro,Mean_Gyro,Std_Gyro,High_Accel_Rate,High_Gyro_Rate,Trip_Count
0,V01,0.494907,0.757024,0.122926,0.148481,29.445111,51.357618,5.442126,9.350456,0.060890,0.065574,15
1,V02,0.380486,0.597780,0.128973,0.111654,16.350469,44.948347,4.920702,6.963669,0.029345,0.083521,16
2,V03,0.609615,0.736601,0.150214,0.187903,28.911927,50.827764,5.390473,9.010858,0.114173,0.064961,16
3,V04,0.380365,0.600559,0.097189,0.114784,25.854231,46.188237,5.144587,8.308738,0.029474,0.058947,15
4,V05,0.142751,0.382663,0.077300,0.069323,6.457722,41.288460,4.008126,5.486347,0.008299,0.031120,15


In [20]:
vehicle_features.describe().T

,count,mean,std,min,25%,50%,75%,max
P95_Accel_Horizontal,30.0,0.397685,0.174491,0.142751,0.198369,0.427519,0.550182,0.641340
P99_Accel_Horizontal,30.0,0.641252,0.113144,0.380702,0.595540,0.681910,0.732952,0.763440
Mean_Accel_Horizontal,30.0,0.112045,0.024705,0.076940,0.093502,0.105341,0.127461,0.160259
Std_Accel_Horizontal,30.0,0.130350,0.036484,0.069323,0.098559,0.127005,0.163933,0.197637
P95_Gyro,30.0,14.463837,10.065445,5.769172,6.293561,7.520658,25.764854,32.722102
P99_Gyro,30.0,44.184303,6.432924,22.599570,42.227375,45.971265,48.573611,51.761746
Mean_Gyro,30.0,4.689411,0.594952,3.629004,4.098330,4.772243,5.185247,5.574218
Std_Gyro,30.0,7.283612,1.507779,4.451791,5.754882,7.709630,8.544164,9.360922
High_Accel_Rate,30.0,0.048902,0.032859,0.005602,0.024999,0.040213,0.072459,0.121076
High_Gyro_Rate,30.0,0.049395,0.018588,0.014006,0.035058,0.050636,0.064004,0.083521


In [22]:
vehicle_features[
    [
        "P95_Accel_Horizontal",
        "P99_Accel_Horizontal",
        "Mean_Accel_Horizontal",
        "Std_Accel_Horizontal",
        "P95_Gyro",
        "P99_Gyro",
        "Mean_Gyro",
        "Std_Gyro",
        "High_Accel_Rate",
        "High_Gyro_Rate",
    ]
].corr().round(3)

,P95_Accel_Horizontal,P99_Accel_Horizontal,Mean_Accel_Horizontal,Std_Accel_Horizontal,P95_Gyro,P99_Gyro,Mean_Gyro,Std_Gyro,High_Accel_Rate,High_Gyro_Rate
P95_Accel_Horizontal,1.000,0.870,0.916,0.968,0.728,0.758,0.918,0.944,0.925,0.833
P99_Accel_Horizontal,0.870,1.000,0.761,0.890,0.582,0.644,0.774,0.797,0.807,0.687
Mean_Accel_Horizontal,0.916,0.761,1.000,0.935,0.810,0.705,0.858,0.852,0.948,0.848
Std_Accel_Horizontal,0.968,0.890,0.935,1.000,0.734,0.689,0.852,0.883,0.978,0.753
P95_Gyro,0.728,0.582,0.810,0.734,1.000,0.603,0.847,0.806,0.772,0.796
P99_Gyro,0.758,0.644,0.705,0.689,0.603,1.000,0.819,0.832,0.673,0.804
Mean_Gyro,0.918,0.774,0.858,0.852,0.847,0.819,1.000,0.982,0.816,0.930
Std_Gyro,0.944,0.797,0.852,0.883,0.806,0.832,0.982,1.000,0.842,0.881
High_Accel_Rate,0.925,0.807,0.948,0.978,0.772,0.673,0.816,0.842,1.000,0.725
High_Gyro_Rate,0.833,0.687,0.848,0.753,0.796,0.804,0.930,0.881,0.725,1.000


In [23]:
selected_vehicle_metrics = [
    "P95_Accel_Horizontal",
    "Std_Accel_Horizontal",
    "P95_Gyro",
    "High_Gyro_Rate",
]

vehicle_features[
    ["Vehicle_ID"] + selected_vehicle_metrics
].sort_values(
    "P95_Accel_Horizontal",
    ascending=False
)

,Vehicle_ID,P95_Accel_Horizontal,Std_Accel_Horizontal,P95_Gyro,High_Gyro_Rate
13,V14,0.641340,0.197637,28.535120,0.062780
22,V23,0.634511,0.182669,32.722102,0.073874
23,V24,0.623241,0.176776,14.846746,0.060096
11,V12,0.620418,0.174334,25.496724,0.066667
2,V03,0.609615,0.187903,28.911927,0.064961
5,V06,0.595013,0.175781,30.865425,0.064078
19,V20,0.590659,0.169079,9.550003,0.055263
18,V19,0.568607,0.170602,27.902087,0.074866
0,V01,0.494907,0.148481,29.445111,0.065574
24,V25,0.487242,0.141952,20.326092,0.063781


In [24]:
vehicle_features[selected_vehicle_metrics].describe().T

,count,mean,std,min,25%,50%,75%,max
P95_Accel_Horizontal,30.0,0.397685,0.174491,0.142751,0.198369,0.427519,0.550182,0.641340
Std_Accel_Horizontal,30.0,0.130350,0.036484,0.069323,0.098559,0.127005,0.163933,0.197637
P95_Gyro,30.0,14.463837,10.065445,5.769172,6.293561,7.520658,25.764854,32.722102
High_Gyro_Rate,30.0,0.049395,0.018588,0.014006,0.035058,0.050636,0.064004,0.083521


In [27]:
vehicle_features[
    [
        "P95_Accel_Horizontal",
        "P95_Gyro",
        "High_Gyro_Rate"
    ]
].corr().round(3)

,P95_Accel_Horizontal,P95_Gyro,High_Gyro_Rate
P95_Accel_Horizontal,1.000,0.728,0.833
P95_Gyro,0.728,1.000,0.796
High_Gyro_Rate,0.833,0.796,1.000


## Vehicle Inspection Score

In [28]:
selected_vehicle_metrics = [
    "P95_Accel_Horizontal",
    "P95_Gyro",
    "High_Gyro_Rate",
]

In [29]:
def fleet_rank_score(series):
    n = len(series)

    if n <= 1 or series.nunique() == 1:
        return pd.Series(5.0, index=series.index)

    return (
        (series.rank(method="average") - 1)
        / (n - 1)
        * 10
    )


for metric in selected_vehicle_metrics:
    vehicle_features[f"{metric}_Score"] = fleet_rank_score(
        vehicle_features[metric]
    )

vehicle_features[
    [
        "Vehicle_ID",
        "P95_Accel_Horizontal_Score",
        "P95_Gyro_Score",
        "High_Gyro_Rate_Score",
    ]
].sort_values(
    "P95_Accel_Horizontal_Score",
    ascending=False
)

,Vehicle_ID,P95_Accel_Horizontal_Score,P95_Gyro_Score,High_Gyro_Rate_Score
13,V14,10.000000,8.620690,6.896552
22,V23,9.655172,10.000000,9.310345
23,V24,9.310345,6.206897,6.206897
11,V12,8.965517,7.241379,8.620690
2,V03,8.620690,8.965517,7.931034
5,V06,8.275862,9.655172,7.586207
19,V20,7.931034,5.517241,5.517241
18,V19,7.586207,8.275862,9.655172
0,V01,7.241379,9.310345,8.275862
24,V25,6.896552,6.896552,7.241379


### Overall Score

In [30]:
vehicle_features["Inspection_Score"] = (
    vehicle_features["P95_Accel_Horizontal_Score"]
    + vehicle_features["P95_Gyro_Score"]
    + vehicle_features["High_Gyro_Rate_Score"]
) / 3

vehicle_features[
    [
        "Vehicle_ID",
        "P95_Accel_Horizontal_Score",
        "P95_Gyro_Score",
        "High_Gyro_Rate_Score",
        "Inspection_Score",
    ]
].sort_values(
    "Inspection_Score",
    ascending=False
)

,Vehicle_ID,P95_Accel_Horizontal_Score,P95_Gyro_Score,High_Gyro_Rate_Score,Inspection_Score
22,V23,9.655172,10.000000,9.310345,9.655172
13,V14,10.000000,8.620690,6.896552,8.505747
18,V19,7.586207,8.275862,9.655172,8.505747
5,V06,8.275862,9.655172,7.586207,8.505747
2,V03,8.620690,8.965517,7.931034,8.505747
0,V01,7.241379,9.310345,8.275862,8.275862
11,V12,8.965517,7.241379,8.620690,8.275862
14,V15,6.551724,7.931034,8.965517,7.816092
23,V24,9.310345,6.206897,6.206897,7.241379
24,V25,6.896552,6.896552,7.241379,7.011494


In [31]:
vehicle_features["Inspection_Score"].agg(
    ["min", "max", "mean", "median", "std"]
)

min       0.459770
max       9.655172
mean      5.000000
median    5.114943
std       2.900900
Name: Inspection_Score, dtype: float64

In [32]:
vehicle_features[
    [
        "P95_Accel_Horizontal_Score",
        "P95_Gyro_Score",
        "High_Gyro_Rate_Score",
        "Inspection_Score",
    ]
].agg(["min", "max"])

,P95_Accel_Horizontal_Score,P95_Gyro_Score,High_Gyro_Rate_Score,Inspection_Score
min,0.0,0.0,0.0,0.459770
max,10.0,10.0,10.0,9.655172


## Weight Sensitivity

Check whether the vehicle ranking changes when the metric weights are changed.

In [33]:
weight_scenarios = {
    "Balanced": {
        "Accel": 1/3,
        "P95_Gyro": 1/3,
        "High_Gyro_Rate": 1/3,
    },
    "Accel_Emphasis": {
        "Accel": 0.40,
        "P95_Gyro": 0.30,
        "High_Gyro_Rate": 0.30,
    },
    "Gyro_Emphasis": {
        "Accel": 0.30,
        "P95_Gyro": 0.40,
        "High_Gyro_Rate": 0.30,
    },
    "Frequency_Emphasis": {
        "Accel": 0.30,
        "P95_Gyro": 0.30,
        "High_Gyro_Rate": 0.40,
    },
}

for name, weights in weight_scenarios.items():
    vehicle_features[f"Score_{name}"] = (
        vehicle_features["P95_Accel_Horizontal_Score"]
        * weights["Accel"]
        +
        vehicle_features["P95_Gyro_Score"]
        * weights["P95_Gyro"]
        +
        vehicle_features["High_Gyro_Rate_Score"]
        * weights["High_Gyro_Rate"]
    )

vehicle_features[
    [
        "Vehicle_ID",
        "Score_Balanced",
        "Score_Accel_Emphasis",
        "Score_Gyro_Emphasis",
        "Score_Frequency_Emphasis",
    ]
].sort_values(
    "Score_Balanced",
    ascending=False
)

,Vehicle_ID,Score_Balanced,Score_Accel_Emphasis,Score_Gyro_Emphasis,Score_Frequency_Emphasis
22,V23,9.655172,9.655172,9.689655,9.620690
5,V06,8.505747,8.482759,8.620690,8.413793
13,V14,8.505747,8.655172,8.517241,8.344828
2,V03,8.505747,8.517241,8.551724,8.448276
18,V19,8.505747,8.413793,8.482759,8.620690
0,V01,8.275862,8.172414,8.379310,8.275862
11,V12,8.275862,8.344828,8.172414,8.310345
14,V15,7.816092,7.689655,7.827586,7.931034
23,V24,7.241379,7.448276,7.137931,7.137931
24,V25,7.011494,7.000000,7.000000,7.034483


## Ranking Stability

In [34]:
scenario_columns = [
    "Score_Balanced",
    "Score_Accel_Emphasis",
    "Score_Gyro_Emphasis",
    "Score_Frequency_Emphasis",
]

ranking_comparison = {}

for column in scenario_columns:
    ranking_comparison[column] = (
        vehicle_features
        .sort_values(column, ascending=False)
        .head(10)["Vehicle_ID"]
        .tolist()
    )

pd.DataFrame(ranking_comparison)

,Score_Balanced,Score_Accel_Emphasis,Score_Gyro_Emphasis,Score_Frequency_Emphasis
0,V23,V23,V23,V23
1,V06,V14,V06,V19
2,V14,V03,V03,V03
3,V03,V06,V14,V06
4,V19,V19,V19,V14
5,V01,V12,V01,V12
6,V12,V01,V12,V01
7,V15,V15,V15,V15
8,V24,V24,V24,V24
9,V25,V25,V25,V02


In [35]:
baseline_rank = (
    vehicle_features
    .sort_values("Score_Balanced", ascending=False)
    .reset_index(drop=True)
)

baseline_rank["Baseline_Rank"] = baseline_rank.index + 1

rank_stability = []

for column in scenario_columns[1:]:
    scenario_rank = (
        vehicle_features
        .sort_values(column, ascending=False)
        .reset_index(drop=True)
    )

    scenario_rank["Scenario_Rank"] = scenario_rank.index + 1

    comparison = baseline_rank[
        ["Vehicle_ID", "Baseline_Rank"]
    ].merge(
        scenario_rank[
            ["Vehicle_ID", "Scenario_Rank"]
        ],
        on="Vehicle_ID"
    )

    rank_stability.append({
        "Scenario": column,
        "Spearman_Correlation": comparison[
            ["Baseline_Rank", "Scenario_Rank"]
        ].corr(method="spearman").iloc[0, 1],
        "Top_10_Overlap": len(
            set(baseline_rank.head(10)["Vehicle_ID"])
            &
            set(scenario_rank.head(10)["Vehicle_ID"])
        ),
    })

pd.DataFrame(rank_stability)

,Scenario,Spearman_Correlation,Top_10_Overlap
0,Score_Accel_Emphasis,0.996440,10
1,Score_Gyro_Emphasis,0.997775,10
2,Score_Frequency_Emphasis,0.993771,9


## Metric Sensitivity

Check whether the ranking changes significantly when one metric is removed.

In [36]:
metric_score_columns = {
    "P95_Accel": "P95_Accel_Horizontal_Score",
    "P95_Gyro": "P95_Gyro_Score",
    "High_Gyro_Rate": "High_Gyro_Rate_Score",
}

metric_sensitivity = {}

for removed_metric, removed_column in metric_score_columns.items():

    remaining_columns = [
        column
        for column in metric_score_columns.values()
        if column != removed_column
    ]

    metric_sensitivity[removed_metric] = (
        vehicle_features[remaining_columns].mean(axis=1)
    )

for name, scores in metric_sensitivity.items():
    vehicle_features[f"Score_without_{name}"] = scores

vehicle_features[
    [
        "Vehicle_ID",
        "Inspection_Score",
        "Score_without_P95_Accel",
        "Score_without_P95_Gyro",
        "Score_without_High_Gyro_Rate",
    ]
].sort_values(
    "Inspection_Score",
    ascending=False
)

,Vehicle_ID,Inspection_Score,Score_without_P95_Accel,Score_without_P95_Gyro,Score_without_High_Gyro_Rate
22,V23,9.655172,9.655172,9.482759,9.827586
13,V14,8.505747,7.758621,8.448276,9.310345
18,V19,8.505747,8.965517,8.620690,7.931034
5,V06,8.505747,8.620690,7.931034,8.965517
2,V03,8.505747,8.448276,8.275862,8.793103
0,V01,8.275862,8.793103,7.758621,8.275862
11,V12,8.275862,7.931034,8.793103,8.103448
14,V15,7.816092,8.448276,7.758621,7.241379
23,V24,7.241379,6.206897,7.758621,7.758621
24,V25,7.011494,7.068966,7.068966,6.896552


## Metric Ranking Stability

In [37]:
metric_scenario_columns = [
    "Inspection_Score",
    "Score_without_P95_Accel",
    "Score_without_P95_Gyro",
    "Score_without_High_Gyro_Rate",
]

metric_ranking = {}

for column in metric_scenario_columns:
    metric_ranking[column] = (
        vehicle_features
        .sort_values(column, ascending=False)
        .head(10)["Vehicle_ID"]
        .tolist()
    )

pd.DataFrame(metric_ranking)

,Inspection_Score,Score_without_P95_Accel,Score_without_P95_Gyro,Score_without_High_Gyro_Rate
0,V23,V23,V23,V23
1,V14,V19,V12,V14
2,V19,V01,V19,V06
3,V06,V06,V14,V03
4,V03,V03,V03,V01
5,V01,V15,V06,V12
6,V12,V02,V01,V19
7,V15,V12,V24,V24
8,V24,V14,V15,V15
9,V25,V25,V25,V25


In [38]:
baseline_rank = (
    vehicle_features
    .sort_values("Inspection_Score", ascending=False)
    .reset_index(drop=True)
)

baseline_rank["Baseline_Rank"] = baseline_rank.index + 1

metric_rank_stability = []

for column in metric_scenario_columns[1:]:
    scenario_rank = (
        vehicle_features
        .sort_values(column, ascending=False)
        .reset_index(drop=True)
    )

    scenario_rank["Scenario_Rank"] = scenario_rank.index + 1

    comparison = baseline_rank[
        ["Vehicle_ID", "Baseline_Rank"]
    ].merge(
        scenario_rank[
            ["Vehicle_ID", "Scenario_Rank"]
        ],
        on="Vehicle_ID"
    )

    metric_rank_stability.append({
        "Scenario": column,
        "Spearman_Correlation": comparison[
            ["Baseline_Rank", "Scenario_Rank"]
        ].corr(method="spearman").iloc[0, 1],
        "Top_10_Overlap": len(
            set(baseline_rank.head(10)["Vehicle_ID"])
            &
            set(scenario_rank.head(10)["Vehicle_ID"])
        ),
    })

pd.DataFrame(metric_rank_stability)

,Scenario,Spearman_Correlation,Top_10_Overlap
0,Score_without_P95_Accel,0.971524,9
1,Score_without_P95_Gyro,0.986652,10
2,Score_without_High_Gyro_Rate,0.988877,10


## Driver-Vehicle Check

Check whether unusual vehicle sensor behaviour could be explained by driver assignment.

In [39]:
driver_vehicle_counts = (
    trips.groupby("Vehicle_ID")["Driver_ID"]
    .nunique()
    .sort_values(ascending=False)
)

driver_vehicle_counts.describe()

count    30.000000
mean      1.466667
std       0.730297
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       4.000000
Name: Driver_ID, dtype: float64

In [40]:
driver_vehicle_counts

Vehicle_ID
V23    4
V09    3
V06    2
V14    2
V10    2
V02    2
V19    2
V28    2
V21    2
V29    2
V03    2
V05    1
V01    1
V04    1
V13    1
V12    1
V11    1
V07    1
V08    1
V18    1
V15    1
V16    1
V22    1
V20    1
V17    1
V24    1
V26    1
V25    1
V27    1
V30    1
Name: Driver_ID, dtype: int64

In [41]:
vehicle_driver_map = (
    trips.groupby("Vehicle_ID")["Driver_ID"]
    .agg(lambda x: ", ".join(sorted(x.unique())))
    .reset_index()
)

vehicle_driver_map

,Vehicle_ID,Driver_ID
0,V01,D01
1,V02,"D02, D29"
2,V03,"D03, D26"
3,V04,D04
4,V05,D05
5,V06,"D06, D30"
6,V07,D07
7,V08,D08
8,V09,"D09, D29, D30"
9,V10,"D10, D29"


### Interpretation

Most vehicles are associated with a single driver, so driver behaviour cannot always be separated from vehicle-level sensor behaviour.

However, the highest-ranked vehicle, V23, is associated with four different drivers. This reduces the likelihood that its unusually high sensor signature is attributable to a single driver's behaviour alone.

Because driver assignments are not balanced across all vehicles, the vehicle health score should therefore be interpreted as an anomaly-based inspection priority rather than a causal measure of mechanical condition.